[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_05/notebook_5_3_cvd_risk_calibration.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 5.3: CVD Risk Prediction and Calibration

**Chapter 5: Evaluation Metrics - Implementing Journey 6 (David)**

**Journey Connection**: This notebook implements Journey 6 from Chapter 3, where David questioned whether marginal improvements in AUROC justified switching from Framingham to a complex ML model. For the clinical context and patient story, refer to Chapter 3.

## Learning Objectives

By the end of this notebook, you will be able to:
1. Implement classical risk scores (Framingham) vs ML models
2. Calculate calibration metrics (Brier score, calibration curves, ECE)
3. Understand discrimination (AUROC) vs calibration trade-offs
4. Apply recalibration methods (Platt scaling, isotonic regression)
5. Understand when marginal AUROC improvements matter clinically

## Clinical Context

Cardiovascular disease (CVD) risk prediction guides treatment decisions for millions. Classical scores like Framingham have been used for decades. ML promises improvements, but at what cost?

**David's story (Chapter 3)**: 58-year-old man, total cholesterol 220 mg/dL, HDL 45 mg/dL, BP 138/88 mmHg, non-smoker, no diabetes, father had an MI at 62. Framingham: **12.4%** 10-year CVD risk. ML model (with longitudinal EHR data): **14.8%**. Both are in the moderate-risk category, and statin therapy is recommended either way: the clinical decision is identical. Was the extra complexity worth it?

**The challenge** (numbers from the book, Chapters 3 and 5):
- Framingham AUROC: 0.76
- ML AUROC: 0.78 (marginal improvement)
- But on external data the ML model's probabilities systematically overestimated risk
- For threshold-based decisions, calibration matters as much as discrimination

The numbers this notebook prints come from a synthetic run and will not match the book's.

---

## Setup

In [1]:
# ── Setup (JupyterLite) ─────────────────────────────────────────────────────
import sys
if sys.platform == "emscripten":          # JupyterLite / Pyodide
    import micropip
    await micropip.install(["seaborn"])

In [2]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, roc_curve, brier_score_loss,
    confusion_matrix, accuracy_score
)
from sklearn.isotonic import IsotonicRegression
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Generate Synthetic CVD Risk Data

We'll create synthetic patient data with CVD risk factors:
- Age, sex, smoking status
- Cholesterol, HDL, blood pressure
- Diabetes, family history
- 10-year CVD outcome

In reality, you would use datasets like Framingham Heart Study, MESA, UK Biobank.

In [3]:
def generate_cvd_risk_data(n_patients=5000, event_rate=0.10, baseline_shift=0.0,
                           fh_prevalence=0.30, fh_effect=0.17):
    """
    Generate synthetic CVD risk data.

    Risk factors based on Framingham:
    - Age, sex, smoking, diabetes
    - Total cholesterol, HDL cholesterol
    - Systolic blood pressure, BP treatment

    baseline_shift, fh_prevalence, fh_effect let us simulate a different population
    (used for external validation in Section 8); the defaults give the development cohort.
    """

    data = []

    for i in range(n_patients):
        # Demographics
        age = np.random.normal(58, 12)
        age = np.clip(age, 30, 80)

        sex = np.random.choice(['M', 'F'])

        # Risk factors
        smoker = np.random.rand() < 0.15  # 15% smokers
        diabetes = np.random.rand() < 0.12  # 12% diabetic
        family_history = np.random.rand() < fh_prevalence  # 30% family history by default

        # Continuous risk factors
        total_chol = np.random.normal(200, 40)
        hdl_chol = np.random.normal(50, 15)
        sbp = np.random.normal(130, 20)
        bp_treatment = np.random.rand() < 0.25  # 25% on BP meds

        # Additional factors (not in Framingham, but ML might use)
        bmi = np.random.normal(27, 5)
        exercise = np.random.choice(['none', 'low', 'moderate', 'high'],
                                    p=[0.3, 0.3, 0.3, 0.1])

        # Calculate risk using Framingham-like formula
        # This is simplified for educational purposes
        log_risk = (
            0.07 * (age - 55) +  # Age effect
            0.3 * (1 if sex == 'M' else 0) +  # Male risk
            0.6 * smoker +  # Smoking
            0.7 * diabetes +  # Diabetes
            0.01 * (total_chol - 200) +  # Total cholesterol
            -0.02 * (hdl_chol - 50) +  # HDL (protective)
            0.015 * (sbp - 120) +  # SBP
            0.3 * bp_treatment +  # On treatment
            fh_effect * family_history +  # Family history (not in Framingham; OR ~1.2)
            0.10 * (bmi - 27) +  # BMI (not in this Framingham score; gives ML real extra information)
            baseline_shift  # Population baseline risk (0 = development cohort)
        )

        # Convert to probability (10-year risk)
        prob_cvd = 1 / (1 + np.exp(-log_risk + 3.09))  # Intercept: ~12-13% 10-year event rate

        # Determine outcome
        cvd_event = np.random.rand() < prob_cvd

        data.append({
            'age': age,
            'sex': sex,
            'smoker': smoker,
            'diabetes': diabetes,
            'family_history': family_history,
            'total_chol': total_chol,
            'hdl_chol': hdl_chol,
            'sbp': sbp,
            'bp_treatment': bp_treatment,
            'bmi': bmi,
            'exercise': exercise,
            'true_risk': prob_cvd,
            'cvd_event': cvd_event
        })

    return pd.DataFrame(data)

# Generate dataset
print("Generating synthetic CVD risk data...")
df = generate_cvd_risk_data(n_patients=5000, event_rate=0.10)

print(f"\nDataset: {len(df)} patients")
print(f"CVD events: {df['cvd_event'].sum()} ({df['cvd_event'].mean()*100:.1f}%)")
print(f"\nRisk factor prevalence:")
print(f"  Smoker:          {df['smoker'].mean()*100:.0f}%")
print(f"  Diabetes:        {df['diabetes'].mean()*100:.0f}%")
print(f"  Family history:  {df['family_history'].mean()*100:.0f}%")
print(f"  BP treatment:    {df['bp_treatment'].mean()*100:.0f}%")

df.head()

Generating synthetic CVD risk data...

Dataset: 5000 patients
CVD events: 674 (13.5%)

Risk factor prevalence:
  Smoker:          15%
  Diabetes:        12%
  Family history:  30%
  BP treatment:    26%


,age,sex,smoker,diabetes,family_history,total_chol,hdl_chol,sbp,bp_treatment,bmi,exercise,true_risk,cvd_event
0,63.960570,M,False,False,False,194.469428,48.580685,111.423438,False,28.962899,none,0.106973,False
1,46.849784,F,False,False,True,209.678491,21.300796,95.501643,False,24.188562,none,0.030226,False
2,49.559874,M,False,False,False,114.415174,67.015089,127.905089,False,25.533004,high,0.012187,False
3,57.641937,F,False,True,False,159.137912,47.573669,119.327024,False,26.972361,none,0.070567,False
4,60.506363,F,False,True,False,121.613195,17.324986,109.122072,False,22.464067,low,0.059992,False


In [4]:
# Visualize risk factors
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Age distribution by outcome
ax = axes[0, 0]
ax.hist(df[df['cvd_event']==False]['age'], bins=30, alpha=0.6, label='No CVD', color='blue')
ax.hist(df[df['cvd_event']==True]['age'], bins=30, alpha=0.6, label='CVD', color='red')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Frequency')
ax.set_title('Age Distribution')
ax.legend()

# Cholesterol
ax = axes[0, 1]
ax.hist(df[df['cvd_event']==False]['total_chol'], bins=30, alpha=0.6, label='No CVD', color='blue')
ax.hist(df[df['cvd_event']==True]['total_chol'], bins=30, alpha=0.6, label='CVD', color='red')
ax.set_xlabel('Total Cholesterol (mg/dL)')
ax.set_ylabel('Frequency')
ax.set_title('Cholesterol Distribution')
ax.legend()

# Blood pressure
ax = axes[0, 2]
ax.hist(df[df['cvd_event']==False]['sbp'], bins=30, alpha=0.6, label='No CVD', color='blue')
ax.hist(df[df['cvd_event']==True]['sbp'], bins=30, alpha=0.6, label='CVD', color='red')
ax.set_xlabel('Systolic BP (mmHg)')
ax.set_ylabel('Frequency')
ax.set_title('Blood Pressure Distribution')
ax.legend()

# Risk factor prevalence by outcome
ax = axes[1, 0]
risk_factors = ['smoker', 'diabetes', 'family_history', 'bp_treatment']
prev_no_cvd = [df[df['cvd_event']==False][rf].mean() for rf in risk_factors]
prev_cvd = [df[df['cvd_event']==True][rf].mean() for rf in risk_factors]
x = np.arange(len(risk_factors))
width = 0.35
ax.bar(x - width/2, prev_no_cvd, width, label='No CVD', color='blue', alpha=0.6)
ax.bar(x + width/2, prev_cvd, width, label='CVD', color='red', alpha=0.6)
ax.set_ylabel('Prevalence')
ax.set_title('Risk Factor Prevalence')
ax.set_xticks(x)
ax.set_xticklabels(['Smoker', 'Diabetes', 'Family Hx', 'BP Rx'], rotation=45, ha='right')
ax.legend()

# True risk distribution
ax = axes[1, 1]
ax.hist(df[df['cvd_event']==False]['true_risk'], bins=30, alpha=0.6, label='No CVD', color='blue')
ax.hist(df[df['cvd_event']==True]['true_risk'], bins=30, alpha=0.6, label='CVD', color='red')
ax.set_xlabel('True 10-Year CVD Risk')
ax.set_ylabel('Frequency')
ax.set_title('True Risk Distribution')
ax.legend()

# Sex distribution
ax = axes[1, 2]
sex_no_cvd = df[df['cvd_event']==False]['sex'].value_counts()
sex_cvd = df[df['cvd_event']==True]['sex'].value_counts()
x = np.arange(2)
ax.bar(x - width/2, [sex_no_cvd['M'], sex_no_cvd['F']], width, label='No CVD', color='blue', alpha=0.6)
ax.bar(x + width/2, [sex_cvd['M'], sex_cvd['F']], width, label='CVD', color='red', alpha=0.6)
ax.set_ylabel('Count')
ax.set_title('Sex Distribution')
ax.set_xticks(x)
ax.set_xticklabels(['Male', 'Female'])
ax.legend()

plt.tight_layout()
plt.show()

print("\n📊 Observation: Classic CVD risk factors clearly associated with outcomes")


📊 Observation: Classic CVD risk factors clearly associated with outcomes


## 2. Implement Framingham Risk Score

Classical risk score based on logistic regression coefficients from the original Framingham Heart Study.

In [5]:
def framingham_risk_score(row):
    """
    Simplified Framingham Risk Score.

    Returns 10-year CVD risk probability.
    """

    # Coefficients (simplified, for educational purposes)
    log_risk = (
        0.07 * (row['age'] - 55) +
        0.3 * (1 if row['sex'] == 'M' else 0) +
        0.6 * row['smoker'] +
        0.7 * row['diabetes'] +
        0.01 * (row['total_chol'] - 200) +
        -0.02 * (row['hdl_chol'] - 50) +
        0.015 * (row['sbp'] - 120) +
        0.3 * row['bp_treatment']
    )

    # Convert to probability
    # Intercept calibrated to the development population (absorbs the average family-history effect)
    risk = 1 / (1 + np.exp(-log_risk + 3.04))

    return risk

# Calculate Framingham scores
df['framingham_risk'] = df.apply(framingham_risk_score, axis=1)

print("Framingham Risk Score calculated")
print(f"Mean predicted risk: {df['framingham_risk'].mean()*100:.1f}%")
print(f"Actual event rate:   {df['cvd_event'].mean()*100:.1f}%")

Framingham Risk Score calculated
Mean predicted risk: 12.8%
Actual event rate:   13.5%


## 3. Train ML Models

Train modern ML models and compare to Framingham.

In [6]:
# Prepare features
feature_cols = ['age', 'sex', 'smoker', 'diabetes', 'family_history',
               'total_chol', 'hdl_chol', 'sbp', 'bp_treatment', 'bmi']

# Encode categorical
df_ml = df.copy()
df_ml['sex'] = (df_ml['sex'] == 'M').astype(int)
df_ml['smoker'] = df_ml['smoker'].astype(int)
df_ml['diabetes'] = df_ml['diabetes'].astype(int)
df_ml['family_history'] = df_ml['family_history'].astype(int)
df_ml['bp_treatment'] = df_ml['bp_treatment'].astype(int)

# Split data
X = df_ml[feature_cols].values
y = df_ml['cvd_event'].values

# Split row indices together with X and y so Framingham scores stay aligned with the same patients
X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X, y, np.arange(len(df_ml)), test_size=0.3, random_state=42, stratify=y
)

# Also get Framingham predictions (same patients as X_train / X_test)

framingham_train = df.iloc[train_idx]['framingham_risk'].values
framingham_test = df.iloc[test_idx]['framingham_risk'].values

print(f"Training set: {len(X_train)} patients ({y_train.mean()*100:.1f}% events)")
print(f"Test set:     {len(X_test)} patients ({y_test.mean()*100:.1f}% events)")

# Train models
models = {}

# Logistic Regression (similar to Framingham)
print("\nTraining Logistic Regression...")
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
models['Logistic Regression'] = lr

# Random Forest
print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
models['Random Forest'] = rf

# Gradient Boosting
print("Training Gradient Boosting...")
gb = GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
models['Gradient Boosting'] = gb

print("\n✓ All models trained")

Training set: 3500 patients (13.5% events)
Test set:     1500 patients (13.5% events)

Training Logistic Regression...
Training Random Forest...


Training Gradient Boosting...



✓ All models trained


## 4. Compare Discrimination (AUROC)

First, let's look at AUROC - the typical metric reported in papers.

In [7]:
# Calculate predictions
predictions = {}
predictions['Framingham'] = framingham_test

for name, model in models.items():
    predictions[name] = model.predict_proba(X_test)[:, 1]

# Calculate AUROC
print("="*70)
print("DISCRIMINATION: AUROC Comparison")
print("="*70)

aurocs = {}
for name, preds in predictions.items():
    auroc = roc_auc_score(y_test, preds)
    aurocs[name] = auroc
    print(f"{name:25s}: {auroc:.4f}")

print("="*70)

# Calculate improvement over Framingham
baseline_auroc = aurocs['Framingham']
print(f"\nImprovement over Framingham:")
for name, auroc in aurocs.items():
    if name != 'Framingham':
        improvement = auroc - baseline_auroc
        pct_improvement = (improvement / baseline_auroc) * 100
        print(f"  {name:23s}: {improvement:+.4f} ({pct_improvement:+.1f}%)")

best_ml = max((m for m in aurocs if m != 'Framingham'), key=aurocs.get)
gain = aurocs[best_ml] - baseline_auroc
if gain > 0.02:
    print(f"\n💡 Observation: the best ML model ({best_ml}) beats Framingham by {gain:+.3f} AUROC")
elif gain > -0.02:
    print(f"\n💡 Observation: the best ML model ({best_ml}) is within 0.02 AUROC of Framingham ({gain:+.3f})")
else:
    print(f"\n💡 Observation: no ML model beats Framingham (best: {best_ml}, {gain:+.3f})")
print("   Note: the synthetic outcome uses the Framingham variables plus family history and BMI,")
print("   which the ML models see and the Framingham score does not: that is the ML models' only edge.")
print("   Is a difference of this size clinically meaningful? Let's check CALIBRATION...")

DISCRIMINATION: AUROC Comparison
Framingham               : 0.7424
Logistic Regression      : 0.7738
Random Forest            : 0.7433
Gradient Boosting        : 0.7361

Improvement over Framingham:
  Logistic Regression    : +0.0314 (+4.2%)
  Random Forest          : +0.0010 (+0.1%)
  Gradient Boosting      : -0.0062 (-0.8%)

💡 Observation: the best ML model (Logistic Regression) beats Framingham by +0.031 AUROC
   Note: the synthetic outcome uses the Framingham variables plus family history and BMI,
   which the ML models see and the Framingham score does not: that is the ML models' only edge.
   Is a difference of this size clinically meaningful? Let's check CALIBRATION...


In [8]:
# Plot ROC curves
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

colors = {'Framingham': 'blue', 'Logistic Regression': 'green',
         'Random Forest': 'orange', 'Gradient Boosting': 'red'}

for name, preds in predictions.items():
    fpr, tpr, _ = roc_curve(y_test, preds)
    auroc = aurocs[name]
    ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUROC={auroc:.3f})',
           color=colors.get(name, 'gray'))

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves: Discrimination Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

spread = max(aurocs.values()) - min(aurocs.values())
if spread < 0.03:
    print(f"📊 ROC curves are very similar - AUROC spread only {spread:.3f}")
else:
    print(f"📊 AUROC spread {spread:.3f}: look at where the curves separate")

📊 AUROC spread 0.038: look at where the curves separate


## 5. Calibration Analysis

**Calibration**: Do predicted probabilities match observed frequencies?

For risk prediction, calibration is MORE important than discrimination!

In [9]:
def calculate_brier_score(y_true, y_pred_proba):
    """Calculate Brier score (lower is better)."""
    return np.mean((y_true - y_pred_proba) ** 2)

def calculate_ece(y_true, y_pred_proba, n_bins=10):
    """
    Calculate Expected Calibration Error (ECE).

    ECE measures average difference between confidence and accuracy.
    """
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]

    ece = 0.0
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (y_pred_proba > bin_lower) & (y_pred_proba <= bin_upper)
        prop_in_bin = np.mean(in_bin)

        if prop_in_bin > 0:
            accuracy_in_bin = np.mean(y_true[in_bin])
            avg_confidence_in_bin = np.mean(y_pred_proba[in_bin])
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin

    return ece

# Calculate calibration metrics
print("="*70)
print("CALIBRATION: Brier Score and ECE")
print("="*70)

calibration_metrics = {}
for name, preds in predictions.items():
    brier = calculate_brier_score(y_test, preds)
    ece = calculate_ece(y_test, preds, n_bins=10)
    calibration_metrics[name] = {'brier': brier, 'ece': ece}

    print(f"{name:25s}:")
    print(f"  Brier Score: {brier:.4f} (lower is better)")
    print(f"  ECE:         {ece:.4f} (lower is better)")

print("="*70)

print("\n⚠️  KEY FINDING:")
fram_brier = calibration_metrics['Framingham']['brier']
rf_brier = calibration_metrics['Random Forest']['brier']
best_cal = min(calibration_metrics, key=lambda k: calibration_metrics[k]['ece'])
print(f"   Best calibrated (lowest ECE): {best_cal} ({calibration_metrics[best_cal]['ece']:.4f})")
if rf_brier > fram_brier:
    print(f"   Random Forest has a WORSE Brier score than Framingham: {rf_brier:.4f} vs {fram_brier:.4f}")
    if aurocs['Random Forest'] > aurocs['Framingham']:
        print(f"   ...despite its higher AUROC: ranking patients well is not the same as accurate risks")
else:
    print(f"   Random Forest has a better (lower) Brier score than Framingham: {rf_brier:.4f} vs {fram_brier:.4f}")

CALIBRATION: Brier Score and ECE
Framingham               :
  Brier Score: 0.1048 (lower is better)
  ECE:         0.0142 (lower is better)
Logistic Regression      :
  Brier Score: 0.1016 (lower is better)
  ECE:         0.0173 (lower is better)
Random Forest            :
  Brier Score: 0.1049 (lower is better)
  ECE:         0.0121 (lower is better)
Gradient Boosting        :
  Brier Score: 0.1096 (lower is better)
  ECE:         0.0327 (lower is better)

⚠️  KEY FINDING:
   Best calibrated (lowest ECE): Random Forest (0.0121)
   Random Forest has a WORSE Brier score than Framingham: 0.1049 vs 0.1048
   ...despite its higher AUROC: ranking patients well is not the same as accurate risks


In [10]:
# Plot calibration curves
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

models_to_plot = ['Framingham', 'Logistic Regression', 'Random Forest', 'Gradient Boosting']

for idx, name in enumerate(models_to_plot):
    ax = axes[idx // 2, idx % 2]

    preds = predictions[name]

    # Calculate calibration curve
    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_test, preds, n_bins=10, strategy='uniform'
    )

    # Plot
    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect Calibration')
    ax.plot(mean_predicted_value, fraction_of_positives, 'o-', linewidth=2,
           markersize=8, label=name, color=colors.get(name, 'gray'))

    ax.set_xlabel('Mean Predicted Probability', fontsize=11)
    ax.set_ylabel('Fraction of Positives', fontsize=11)
    ax.set_title(f'{name}\nBrier={calibration_metrics[name]["brier"]:.4f}, ECE={calibration_metrics[name]["ece"]:.4f}',
                fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

print("\n📊 Calibration Curve Interpretation:")
print("   - Points on diagonal = well calibrated")
print("   - Above diagonal = underestimates risk")
print("   - Below diagonal = overestimates risk")
for name in models_to_plot:
    print(f"   - {name}: ECE {calibration_metrics[name]['ece']:.3f}")
print(f"   - Best calibrated on this test set: {best_cal}")


📊 Calibration Curve Interpretation:
   - Points on diagonal = well calibrated
   - Above diagonal = underestimates risk
   - Below diagonal = overestimates risk
   - Framingham: ECE 0.014
   - Logistic Regression: ECE 0.017
   - Random Forest: ECE 0.012
   - Gradient Boosting: ECE 0.033
   - Best calibrated on this test set: Random Forest


## 6. Recalibration Methods

Can we fix poorly calibrated models?

In [11]:
# Focus on Random Forest
# Fit the calibrators on OUT-OF-FOLD predictions: the RF has memorised its own training rows,
# so its in-sample predictions are overconfident and would teach the calibrator the wrong mapping.
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
rf_preds_oof = cross_val_predict(clone(rf), X_train, y_train, cv=5, method='predict_proba')[:, 1]
rf_preds_test = rf.predict_proba(X_test)[:, 1]

# Method 1: Platt Scaling (fit logistic regression on predictions)
print("Applying Platt Scaling...")
platt_scaler = LogisticRegression()
platt_scaler.fit(rf_preds_oof.reshape(-1, 1), y_train)
rf_preds_platt = platt_scaler.predict_proba(rf_preds_test.reshape(-1, 1))[:, 1]

# Method 2: Isotonic Regression (non-parametric)
print("Applying Isotonic Regression...")
iso_regressor = IsotonicRegression(out_of_bounds='clip')
iso_regressor.fit(rf_preds_oof, y_train)
rf_preds_isotonic = iso_regressor.predict(rf_preds_test)

# Calculate metrics
print("\n" + "="*70)
print("RECALIBRATION RESULTS: Random Forest")
print("="*70)

recalibrated = {
    'Uncalibrated RF': rf_preds_test,
    'RF + Platt Scaling': rf_preds_platt,
    'RF + Isotonic Regression': rf_preds_isotonic
}

for name, preds in recalibrated.items():
    auroc = roc_auc_score(y_test, preds)
    brier = calculate_brier_score(y_test, preds)
    ece = calculate_ece(y_test, preds)

    print(f"\n{name}:")
    print(f"  AUROC:       {auroc:.4f}")
    print(f"  Brier Score: {brier:.4f}")
    print(f"  ECE:         {ece:.4f}")

print("\n" + "="*70)
ece0, brier0 = calculate_ece(y_test, rf_preds_test), calculate_brier_score(y_test, rf_preds_test)
for name in ['RF + Platt Scaling', 'RF + Isotonic Regression']:
    e, b = calculate_ece(y_test, recalibrated[name]), calculate_brier_score(y_test, recalibrated[name])
    print(f"{'✓' if e < ece0 else '✗'} {name}: ECE {ece0:.4f} → {e:.4f}, Brier {brier0:.4f} → {b:.4f}")
if ece0 < 0.03:
    print("  The uncalibrated RF was already fairly well calibrated here, so there was little to fix.")
print("  Compare the AUROC values above: Platt scaling with a positive slope is strictly increasing, so it")
print("  keeps the ranking (same AUROC); isotonic regression can merge scores into ties and shift AUROC.")
print("  Recalibration aims to fix probabilities, not ranking; always re-check both after recalibrating.")
print("  → Was the complexity worth it?")

Applying Platt Scaling...
Applying Isotonic Regression...

RECALIBRATION RESULTS: Random Forest

Uncalibrated RF:
  AUROC:       0.7433
  Brier Score: 0.1049
  ECE:         0.0121

RF + Platt Scaling:
  AUROC:       0.7433
  Brier Score: 0.1062
  ECE:         0.0324

RF + Isotonic Regression:
  AUROC:       0.7358
  Brier Score: 0.1062
  ECE:         0.0226

✗ RF + Platt Scaling: ECE 0.0121 → 0.0324, Brier 0.1049 → 0.1062
✗ RF + Isotonic Regression: ECE 0.0121 → 0.0226, Brier 0.1049 → 0.1062
  The uncalibrated RF was already fairly well calibrated here, so there was little to fix.
  Compare the AUROC values above: Platt scaling with a positive slope is strictly increasing, so it
  keeps the ranking (same AUROC); isotonic regression can merge scores into ties and shift AUROC.
  Recalibration aims to fix probabilities, not ranking; always re-check both after recalibrating.
  → Was the complexity worth it?


In [12]:
# Plot recalibrated curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, preds) in enumerate(recalibrated.items()):
    ax = axes[idx]

    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_test, preds, n_bins=10, strategy='uniform'
    )

    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect')
    ax.plot(mean_predicted_value, fraction_of_positives, 'o-', linewidth=2,
           markersize=8, color='red')

    # Also show Framingham for comparison
    fram_frac, fram_mean = calibration_curve(y_test, framingham_test, n_bins=10)
    ax.plot(fram_mean, fram_frac, 's-', linewidth=2, markersize=6,
           color='blue', alpha=0.5, label='Framingham')

    brier = calculate_brier_score(y_test, preds)
    ece = calculate_ece(y_test, preds)

    ax.set_xlabel('Mean Predicted Probability', fontsize=11)
    ax.set_ylabel('Fraction of Positives', fontsize=11)
    ax.set_title(f'{name}\nBrier={brier:.4f}, ECE={ece:.4f}',
                fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

improved = [m for m in ['RF + Platt Scaling', 'RF + Isotonic Regression']
            if calculate_ece(y_test, recalibrated[m]) < calculate_ece(y_test, rf_preds_test)]
if improved:
    print(f"\n✓ Closer to the diagonal (lower ECE) after: {', '.join(improved)}")
else:
    print("\n✗ Neither recalibration lowered ECE on this test set")


✗ Neither recalibration lowered ECE on this test set


## 7. David's Case: Clinical Decision-Making

Let's simulate David's specific case.

In [13]:
# David's profile
david = {
    'age': 58,
    'sex': 'M',
    'smoker': False,
    'diabetes': False,
    'family_history': True,
    'total_chol': 220,
    'hdl_chol': 45,
    'sbp': 138,         # BP 138/88; this simplified score uses systolic only
    'bp_treatment': False,
    'bmi': 28
}

print("="*70)
print("JOURNEY 6: DAVID'S CASE")
print("="*70)

print("\nDavid's profile:")
print(f"  Age: {david['age']}")
print(f"  Sex: {david['sex']}")
print(f"  Smoker: {david['smoker']}")
print(f"  Diabetes: {david['diabetes']}")
print(f"  Family history: {david['family_history']}")
print(f"  Total cholesterol: {david['total_chol']} mg/dL")
print(f"  HDL cholesterol: {david['hdl_chol']} mg/dL")
print(f"  Blood pressure: {david['sbp']}/88 mmHg")
print(f"  BMI: {david['bmi']}")

# Calculate Framingham risk
david_df = pd.DataFrame([david])
david_framingham = framingham_risk_score(david_df.iloc[0])

# ML prediction
david_features = [[58, 1, 0, 0, 1, 220, 45, 138, 0, 28]]  # Encoded, same order as feature_cols
# Choose David's "ML model" WITHOUT looking at the test set: 5-fold CV AUROC on the training data only.
# (Picking the best TEST AUROC and then reporting that same test AUROC would be optimistically biased.)
from sklearn.base import clone
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv_auc = {name: cross_val_score(clone(m), X_train, y_train, scoring='roc_auc',
                                cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean()
          for name, m in models.items()}
ml_name = max(cv_auc, key=cv_auc.get)   # frozen choice; its test AUROC below is evaluated once
print("ML model chosen by 5-fold CV on the training set (test set not used):")
for name, a in cv_auc.items():
    print(f"  {name:20s} CV AUROC {a:.3f}{'  ← chosen' if name == ml_name else ''}")
david_ml = models[ml_name].predict_proba(david_features)[0, 1]

print("\n" + "-"*70)
print("Risk Predictions:")
print("-"*70)
print(f"Framingham (classical): {david_framingham*100:.1f}% 10-year CVD risk")
print(f"ML model ({ml_name}): {david_ml*100:.1f}% 10-year CVD risk")
print("-"*70)
print("From the book (Chapter 3): Framingham 12.4%, ML 14.8%; both moderate risk, statin recommended either way")
print("The values above are from this synthetic run.")

# Treatment threshold (typically 7.5-10%)
threshold = 0.075  # 7.5%

print(f"\nTreatment threshold: {threshold*100:.1f}% (ACC/AHA guidelines)")
print(f"\nFramingham: {david_framingham*100:.1f}% → {'TREAT with statins' if david_framingham >= threshold else 'NO treatment'}")
print(f"ML model:   {david_ml*100:.1f}% → {'TREAT with statins' if david_ml >= threshold else 'NO treatment'}")

if (david_framingham >= threshold) != (david_ml >= threshold):
    print("\n⚠️  DISAGREEMENT: Models give different treatment recommendations!")
    print("   Which one should the doctor trust?")
else:
    print("\n✓ Agreement: Both models give same recommendation")
    print(f"  But ML adds {(david_ml - david_framingham)*100:.1f} percentage points to estimated risk")

rf_ece, fram_ece = calibration_metrics[ml_name]['ece'], calibration_metrics['Framingham']['ece']
print("\n💭 David's Question:")
print(f"   'Is the ML model's {(david_ml - david_framingham)*100:+.1f} percentage-point difference accurate?'")
print(f"   'On the test set, ECE was {rf_ece:.3f} for the {ml_name} and {fram_ece:.3f} for Framingham...'")
print("   'Should I trust this prediction?'")

print("\n🩺 What this notebook's numbers say:")
print(f"   - AUROC: Framingham {aurocs['Framingham']:.3f}, {ml_name} {aurocs[ml_name]:.3f}")
print(f"   - ECE:   Framingham {fram_ece:.3f}, {ml_name} {rf_ece:.3f}")
if abs(rf_ece - fram_ece) < 0.005:
    print("   - Calibration is about equally good here; the simpler, validated score is easier to defend")
elif rf_ece > fram_ece:
    print("   - Framingham is better calibrated here: its absolute risk is the safer number to act on")
else:
    print(f"   - The {ml_name} is at least as well calibrated here; trust still depends on external validation")
print("   - For individual risk prediction, calibration > discrimination")
print("   - Real Framingham-type scores have decades of external validation; a new ML model needs the same")
print(f"   Book (Chapter 3/5): Framingham 12.4% vs ML 14.8%; AUROC 0.76 vs 0.78. "
      f"This run: {david_framingham:.1%} vs {david_ml:.1%}; AUROC {aurocs['Framingham']:.2f} vs {aurocs[ml_name]:.2f}")
print(f"   ⚠️ Synthetic data (event rate {df['cvd_event'].mean():.0%}); a single patient's ML estimate also varies")
print("      from run to run, so read these as an illustration, not a clinical estimate.")
print("="*70)

JOURNEY 6: DAVID'S CASE

David's profile:
  Age: 58
  Sex: M
  Smoker: False
  Diabetes: False
  Family history: True
  Total cholesterol: 220 mg/dL
  HDL cholesterol: 45 mg/dL
  Blood pressure: 138/88 mmHg
  BMI: 28


ML model chosen by 5-fold CV on the training set (test set not used):
  Logistic Regression  CV AUROC 0.759  ← chosen
  Random Forest        CV AUROC 0.721
  Gradient Boosting    CV AUROC 0.710

----------------------------------------------------------------------
Risk Predictions:
----------------------------------------------------------------------
Framingham (classical): 12.3% 10-year CVD risk
ML model (Logistic Regression): 15.8% 10-year CVD risk
----------------------------------------------------------------------
From the book (Chapter 3): Framingham 12.4%, ML 14.8%; both moderate risk, statin recommended either way
The values above are from this synthetic run.

Treatment threshold: 7.5% (ACC/AHA guidelines)

Framingham: 12.3% → TREAT with statins
ML model:   15.8% → TREAT with statins

✓ Agreement: Both models give same recommendation
  But ML adds 3.4 percentage points to estimated risk

💭 David's Question:
   'Is the ML model's +3.4 percentage-point difference accurate?'
  

## 8. External Validation: A Population with Lower Baseline Risk

The book's warning about David's ML model is that it **overestimated risk on external data**. A model's calibration
belongs to the population it was fitted on. We simulate a second, synthetic hospital where:

- baseline CVD risk is lower (log-odds shifted by −0.8), and
- family history is recorded much more liberally (60% "yes" instead of 30%, e.g. any relative with any heart disease),
  so it carries little real risk there

The Framingham-style score does not use family history; the ML models learned it. No model is retrained.

In [14]:
np.random.seed(2024)
df_ext = generate_cvd_risk_data(n_patients=3000, baseline_shift=-0.8, fh_prevalence=0.60, fh_effect=0.1)
df_ext_ml = df_ext.copy()
df_ext_ml['sex'] = (df_ext_ml['sex'] == 'M').astype(int)
for col in ['smoker', 'diabetes', 'family_history', 'bp_treatment']:
    df_ext_ml[col] = df_ext_ml[col].astype(int)
X_ext, y_ext = df_ext_ml[feature_cols].values, df_ext_ml['cvd_event'].values

ext_preds = {'Framingham': df_ext.apply(framingham_risk_score, axis=1).values}
for name, model in models.items():
    ext_preds[name] = model.predict_proba(X_ext)[:, 1]

rows = []
for name in ext_preds:
    rows.append({'Model': name,
                 'Internal AUROC': aurocs[name], 'External AUROC': roc_auc_score(y_ext, ext_preds[name]),
                 'Internal ECE': calibration_metrics[name]['ece'], 'External ECE': calculate_ece(y_ext, ext_preds[name]),
                 'Ext. mean predicted': ext_preds[name].mean()})
ext_table = pd.DataFrame(rows)
print(f"External cohort: {len(y_ext)} synthetic patients, observed event rate {y_ext.mean():.1%} "
      f"(development cohort: {df['cvd_event'].mean():.1%})\n")
print(ext_table.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (title, yy, pp) in zip(axes, [('Internal test set', y_test, predictions),
                                      ('External cohort', y_ext, ext_preds)]):
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    for name in ['Framingham', 'Random Forest', 'Gradient Boosting']:
        frac, mean_pred = calibration_curve(yy, pp[name], n_bins=10)
        ax.plot(mean_pred, frac, 'o-', label=name, color=colors.get(name, 'gray'))
    ax.set_xlabel('Mean Predicted Probability'); ax.set_ylabel('Fraction of Positives')
    ax.set_title(title, fontweight='bold'); ax.legend(); ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
plt.tight_layout()
plt.show()

print("\n📊 What the external cohort shows (computed above):")
obs = y_ext.mean()
for _, r in ext_table.iterrows():
    gap = r['Ext. mean predicted'] - obs
    word = 'OVERestimates' if gap > 0.02 else 'UNDERestimates' if gap < -0.02 else 'matches'
    print(f"   {r['Model']:20s} {word} average risk: {r['Ext. mean predicted']:.1%} predicted vs {obs:.1%} observed; "
          f"AUROC {r['Internal AUROC']:.3f} → {r['External AUROC']:.3f}")
ml_names = [m for m in ext_preds if m != 'Framingham']
worst_ml = max(ml_names, key=lambda m: ext_table.set_index('Model').loc[m, 'External ECE'])
fram_ext_ece = ext_table.set_index('Model').loc['Framingham', 'External ECE']
worst_ece = ext_table.set_index('Model').loc[worst_ml, 'External ECE']
if worst_ece > fram_ext_ece:
    print(f"\n   The ML models drift more ({worst_ml}: ECE {worst_ece:.3f} vs Framingham {fram_ext_ece:.3f}): on top of the lower")
    print("   baseline risk, they rely on family history, which means something different at this site.")
else:
    print(f"\n   Here Framingham drifts at least as much as the ML models (ECE {fram_ext_ece:.3f} vs {worst_ece:.3f}).")
if all(ext_table['Ext. mean predicted'] > obs + 0.02):
    print("   Note that EVERY model overestimates in this lower-risk population. Real risk scores are therefore")
    print("   recalibrated to local incidence before use; discrimination (AUROC) changes much less than calibration.")

External cohort: 3000 synthetic patients, observed event rate 7.4% (development cohort: 13.5%)

              Model  Internal AUROC  External AUROC  Internal ECE  External ECE  Ext. mean predicted
         Framingham           0.742           0.753         0.014         0.054                0.129
Logistic Regression           0.774           0.782         0.017         0.068                0.142
      Random Forest           0.743           0.743         0.012         0.064                0.138
  Gradient Boosting           0.736           0.741         0.033         0.054                0.129

📊 What the external cohort shows (computed above):
   Framingham           OVERestimates average risk: 12.9% predicted vs 7.4% observed; AUROC 0.742 → 0.753
   Logistic Regression  OVERestimates average risk: 14.2% predicted vs 7.4% observed; AUROC 0.774 → 0.782
   Random Forest        OVERestimates average risk: 13.8% predicted vs 7.4% observed; AUROC 0.743 → 0.743
   Gradient Boosting    OVERe

## 9. Key Takeaways

### What We Learned

1. **Discrimination vs Calibration**:
   - **AUROC (discrimination)**: Model's ability to rank patients by risk
   - **Calibration**: Predicted probabilities match observed frequencies
   - For risk prediction: **Calibration is MORE important**

2. **Marginal AUROC improvements may not matter**:
   - Compare the AUROC differences in Section 4 with the calibration differences in Section 5
   - A small AUROC gain is worth little if the probabilities are miscalibrated
   - In this synthetic data the Framingham-style score is close to the true model, so ML has little to add

3. **Calibration metrics**:
   - **Brier Score**: Mean squared error of probabilities (lower better)
   - **ECE**: Expected Calibration Error (lower better)
   - **Calibration curve**: Visual check (should be on diagonal)

4. **Recalibration methods**:
   - **Platt Scaling**: Fit logistic regression on predictions
   - **Isotonic Regression**: Non-parametric monotonic mapping
   - Fit them on held-out or out-of-fold predictions, never on the training predictions
   - Platt scaling with a positive slope keeps AUROC; isotonic can change it (ties). Re-check both

5. **Why Journey 6 (David) matters**:
   - ML hype focuses on AUROC
   - But clinicians need calibrated probabilities for decisions
   - Simpler, well-calibrated models may be better than complex poorly-calibrated ones
   - External validation is essential: in Section 8, calibration drifted far more than AUROC

### Connections to Book Chapters

- **Chapter 3 (Seven Journeys)**: David's story provides the clinical motivation
- **Chapter 4 (Data Preparation)**: Training data requirements for calibration
- **Chapter 5 (Evaluation)**: This chapter - calibration metrics, Brier score, ECE
- **Chapter 11 (Deployment)**: Calibration drift over time, monitoring

### Real-World Context

**Clinical risk scores**:
- Framingham (1998): Most widely used, well-validated
- ASCVD Pooled Cohort Equations (2013): Updated US guidelines
- SCORE (Europe), QRISK (UK): Regional variations

**ML attempts**:
- Many papers report improved AUROC (0.75 → 0.78)
- But often poorly calibrated on external data
- Calibration drift when population changes
- Guideline committees remain cautious

**Why calibration is hard**:
- Requires large, representative samples
- Population differences affect calibration more than discrimination
- ML models often overfit → poor calibration
- Temporal drift: risk factors change over time

**Clinical decisions**:
- Treatment thresholds based on absolute risk (7.5%, 10%)
- Require well-calibrated probabilities
- Not just "is risk higher or lower?"
- Shared decision-making with patients

---

## Exercises

1. **Temporal validation**: Split data by time (train on early years, test on later). Does calibration degrade?

2. **Subgroup calibration**: Check calibration separately for men/women, young/old. Are there disparities?

3. **Decision curve analysis**: Plot net benefit across risk thresholds. Does ML improve clinical utility?

4. **Number needed to screen**: Calculate how many patients need to be screened with ML vs Framingham to prevent one event. Worth the cost?

5. **Recalibration-in-the-large**: In Section 8, shift each model's log-odds by a constant so the mean predicted risk matches the external event rate. How much of the ECE gap disappears?

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 5: Evaluation Metrics)*  
*This implements Journey 6 (David - CVD Risk Calibration) from Chapter 3*  
*For clinical context, see Chapter 3. For calibration methods, see Chapter 5.*